# 03 · Una CNN para reconocer dígitos


Ejecuta las celdas en orden. Antes de mirar la solución, intenta la actividad y anota una predicción. Si cambias parámetros, vuelve a ejecutar desde la celda de creación del modelo: llamar a `fit` otra vez continúa el entrenamiento anterior.

**Entorno:** Google Colab o Jupyter con `pip install tensorflow numpy matplotlib scikit-learn notebook`. Reinicia el kernel tras instalar si es necesario. No se necesitan descargas de conjuntos de datos. Los resultados y tiempos pueden variar por versión, CPU y aleatoriedad.

**Tiempo:** 40–45 minutos.

## 1. La arquitectura
Una convolución desplaza filtros aprendidos sobre regiones locales; compartir los mismos pesos en distintas posiciones produce mapas de características. ReLU introduce no linealidad. **Max pooling** resume cada región por su máximo y reduce resolución espacial. **Flatten** transforma los mapas en un vector, y una capa **fully connected (Dense)** combina sus elementos para clasificar. Softmax produce diez puntuaciones normalizadas que suman 1.

En este ejemplo: `8×8×1 → Conv2D(8, 3, same) → 8×8×8 → MaxPool(2) → 4×4×8 → Flatten → 128 → Dense(10)`. Una convolución 3×3 con 1 canal y 8 filtros tiene `(3×3×1+1)×8 = 80` parámetros, incluidos sesgos. Pooling no tiene parámetros entrenables. No todas las CNN necesitan pooling o Flatten.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)

In [ ]:
def curvas(h, metrica="loss"):
    plt.figure(figsize=(7, 4))
    plt.plot(h.history[metrica], label="Entrenamiento")
    plt.plot(h.history["val_" + metrica], label="Validación")
    plt.xlabel("Época")
    plt.ylabel(metrica)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

## 2. Los datos

`load_digits` contiene imágenes reales pequeñas de dígitos escritos a mano. Escalar de 0–16 a 0–1 usa el rango conocido del dataset. La cuarta dimensión es el canal de gris; las etiquetas son enteros de 0 a 9.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

imagenes, etiquetas = load_digits(return_X_y=True)
# Imágenes reales de 8x8 en escala de grises, etiquetas de 0 a 9.
X = (imagenes.reshape(-1, 8, 8, 1) / 16.0).astype("float32")
y = etiquetas.astype("int32")
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print("Entrenamiento, validación, prueba:", X_train.shape, X_val.shape, X_test.shape)

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i, :, :, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(str(y_train[i]))
    ax.axis("off")
plt.show()

## 3. Construye y entrena

Antes de ejecutar: calcula las dimensiones tras cada capa. ¿Por qué `sparse_categorical_crossentropy` es adecuada para etiquetas enteras y una salida de 10 probabilidades?

In [ ]:
modelo = keras.Sequential(
    [
        keras.Input(shape=(8, 8, 1)),
        keras.layers.Conv2D(8, 3, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(2),
        keras.layers.Flatten(),
        keras.layers.Dense(10, activation="softmax"),
    ]
)
modelo.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
modelo.summary()
historia = modelo.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    verbose=0,
)
curvas(historia)
curvas(historia, "accuracy")

La pérdida compara la probabilidad asignada a la clase correcta con la etiqueta entera. La exactitud es la proporción clasificada correctamente. Compara ambas curvas: un aumento de exactitud no garantiza que la pérdida siga mejorando.

## 4. Tu turno: inspecciona errores en validación

Predice las clases de validación con `argmax`; muestra hasta seis imágenes mal clasificadas con etiqueta real y predicción. ¿Qué ambigüedades visuales observas?

In [ ]:
# Tu código aquí.

**Mi interpretación:** escribe aquí.

## 5. Solución de referencia

In [ ]:
prob_val = modelo.predict(X_val, verbose=0)
pred_val = np.argmax(prob_val, axis=1)
errores = np.flatnonzero(pred_val != y_val)
print("Errores de validación:", len(errores), "de", len(y_val))
if len(errores) > 0:
    fig, axes = plt.subplots(1, min(6, len(errores)), figsize=(12, 2), squeeze=False)
    for ax, idx in zip(axes[0], errores[:6]):
        ax.imshow(X_val[idx, :, :, 0], cmap="gray")
        ax.set_title(f"Real {y_val[idx]} / Pred {pred_val[idx]}")
        ax.axis("off")
    plt.show()

Los errores pueden mostrar trazos ambiguos o poco representados; una imagen aislada no demuestra la causa. Usa validación para ajustar la arquitectura y conserva prueba para una evaluación final independiente.

## 6. Evaluación final opcional

Una vez hayas terminado de ajustar el modelo con validación, ejecuta esta celda una sola vez. Si usas prueba para rediseñarlo, reserva otra evaluación independiente.

In [ ]:
resultado = modelo.evaluate(X_test, y_test, verbose=0, return_dict=True)
print("Resultado final de prueba:", resultado)

## 7. Desafío opcional

Prueba 4 filtros en lugar de 8, reinicia y compara número de parámetros, curvas y exactitud de validación. Predice antes cómo cambia el número de parámetros de la primera convolución. No consultes prueba para decidir.